# 1-Shot RLVR with 4-bit Quantization on T4 GPU


## Cell 1: Check GPU

In [ ]:
# ── Check GPU ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU name       : {torch.cuda.get_device_name(0)}')
print(f'Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2: Install Dependencies

In [ ]:
# ── Install all required packages ──────────────────────────────────────────
# Using specific versions to avoid compatibility issues on Colab
!pip install -q \
    peft==0.14.0 \
    bitsandbytes==0.45.0 \
    accelerate==1.2.1 \
    datasets==3.2.0 \
    math-verify \
    sentencepiece

print('✅ All packages installed')

In [ ]:
!pip install -q --upgrade --force-reinstall \
    trl==0.15.2 \
    transformers==4.46.3 \
    accelerate==1.2.1

In [ ]:
!pip install --upgrade --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────
import os
import re
import gc
import json
import torch
import numpy as np
from datetime import datetime
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType
from trl import GRPOConfig, GRPOTrainer

# Suppress minor warnings
import warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Imports successful')
print(f'   PyTorch  : {torch.__version__}')
print(f'   Device   : {torch.cuda.get_device_name(0)}')

2026-05-17 09:59:55.431129: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779011995.815751     230 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779011995.931300     230 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779011996.920901     230 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779011996.920947     230 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779011996.920949     230 computation_placer.cc:177] computation placer alr

✅ Imports successful
   PyTorch  : 2.5.1+cu121
   Device   : Tesla T4


## Cell 4: Configuration
All hyperparameters in one place — tweak here if you get OOM errors.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# NOTE: If you get OOM errors, reduce MAX_NEW_TOKENS or NUM_GENERATIONS first

CFG = dict(
    # Model
    model_name        = 'Qwen/Qwen2.5-Math-1.5B',   # same base model as paper

    # Quantization
    load_in_4bit      = True,
    quant_type        = 'nf4',                        # NormalFloat4 — best for LLMs
    compute_dtype     = torch.float16,
    double_quant      = True,                         # saves extra ~0.4 GB

    # LoRA  (only train a small adapter, not full weights)
    lora_r            = 16,
    lora_alpha        = 32,
    lora_dropout      = 0.05,
    # which layers to adapt (attention + feed-forward)
    lora_target       = ['q_proj','k_proj','v_proj','o_proj',
                         'gate_proj','up_proj','down_proj'],

    # GRPO Training
    # Paper used: batch=128, rollouts=8, response_len=3072, steps=2000
    per_device_batch  = 2,     
    grad_accum        = 1,     # effective batch = 4 (paper used 128)
    num_generations   = 2,     # rollouts per prompt (paper used 8)
    max_new_tokens    = 512,   # response length (paper used 3072)
    max_prompt_len    = 256,
    num_train_steps   = 30,   # paper used 2000
    learning_rate     = 5e-6,
    temperature       = 0.6,   # same as paper
    save_steps        = 50,
    logging_steps     = 10,

    # Dataset replication trick from paper:
    # paper duplicated single example to fill batch of 128
    dataset_repeat    = 64,    # repeat π1 to fill dataset

    # Output
    output_dir        = './oneshot_rlvr_output',
)

print('✅ Configuration set')
print(f"   Model          : {CFG['model_name']}")
print(f"   Quantization   : 4-bit {CFG['quant_type']}")
print(f"   LoRA rank      : {CFG['lora_r']}")
print(f"   Generations    : {CFG['num_generations']} per prompt")
print(f"   Max new tokens : {CFG['max_new_tokens']}")
print(f"   Train steps    : {CFG['num_train_steps']}")

✅ Configuration set
   Model          : Qwen/Qwen2.5-Math-1.5B
   Quantization   : 4-bit nf4
   LoRA rank      : 16
   Generations    : 2 per prompt
   Max new tokens : 512
   Train steps    : 30


## Cell 5: The Training Example (π₁ from Paper)
This is the **exact same** example the paper used — the wind pressure algebra problem.

In [3]:
# ── π₁ — The single training example from the paper (Table 2) ─────────────
# Paper found this single example raises MATH500 from 36% → 73.6%

PI_1_PROMPT = (
    "The pressure P exerted by wind on a sail varies jointly as the area A of the sail "
    "and the cube of the wind's velocity V. "
    "When the velocity is 8 miles per hour, the pressure on a sail of 2 square feet is 4 pounds. "
    "Find the wind velocity when the pressure on 4 square feet of sail is 32 pounds. "
    "Let's think step by step and output the final answer within \\boxed{}."
)
PI_1_ANSWER = "12.8"   # ground truth label from paper

# We also include π₁₃ (geometry example from paper, Table 21)
# to test 2-shot as paper did
PI_13_PROMPT = (
    "Given that circle C passes through points P(0,-4), Q(2,0), and R(3,-1). "
    "(1) Find the equation of circle C. "
    "(2) If the line l: mx+y-1=0 intersects circle C at points A and B, "
    "and |AB|=4, find the value of m. "
    "Let's think step by step and output the final answer within \\boxed{}."
)
PI_13_ANSWER = "4/3"

print('✅ Training examples loaded')
print(f'   π₁  answer: {PI_1_ANSWER}')
print(f'   π₁₃ answer: {PI_13_ANSWER}')
print()
print('π₁ prompt preview:')
print(PI_1_PROMPT[:120], '...')

✅ Training examples loaded
   π₁  answer: 12.8
   π₁₃ answer: 4/3

π₁ prompt preview:
The pressure P exerted by wind on a sail varies jointly as the area A of the sail and the cube of the wind's velocity V. ...


## Cell 6: Build Dataset
Paper trick: duplicate the single example to fill the training batch.

In [4]:
# ── Build 1-shot dataset (paper Section 3.1 trick) ─────────────────────────
# Paper: 'we duplicate the selected example until reaching 128 samples'
# We do the same but with smaller batch size

def build_dataset(prompt: str, answer: str, repeat: int) -> Dataset:
    """Repeat a single example to fill the dataset (same trick as paper)."""
    return Dataset.from_dict({
        'prompt' : [prompt] * repeat,
        'answer' : [answer] * repeat,
    })

# 1-shot dataset using π₁ only
train_dataset_1shot = build_dataset(
    PI_1_PROMPT, PI_1_ANSWER, CFG['dataset_repeat']
)

# 2-shot dataset using π₁ + π₁₃ (alternating)
train_dataset_2shot = Dataset.from_dict({
    'prompt' : ([PI_1_PROMPT, PI_13_PROMPT] * (CFG['dataset_repeat'] // 2)),
    'answer' : ([PI_1_ANSWER, PI_13_ANSWER] * (CFG['dataset_repeat'] // 2)),
})

print(f'✅ Datasets built')
print(f'   1-shot dataset size : {len(train_dataset_1shot)}')
print(f'   2-shot dataset size : {len(train_dataset_2shot)}')
print(f'   (All rows are repeated copies of 1 or 2 examples — same as paper)')

✅ Datasets built
   1-shot dataset size : 64
   2-shot dataset size : 64
   (All rows are repeated copies of 1 or 2 examples — same as paper)


## Cell 7: Reward Function
Binary reward: **1** if answer matches, **0** otherwise — same as paper.

In [5]:
# ── Reward function — binary 0/1 exactly like the paper ───────────────────

def extract_boxed_answer(text: str) -> str:
    """Extract content from \\boxed{...} in model output."""
    # Handle nested braces e.g. \boxed{\frac{4}{3}}
    pattern = r'\\boxed\{([^{}]*)\}'
    matches = re.findall(pattern, text)
    if matches:
        return matches[-1].strip()   # take last boxed answer

    # Fallback: look for boxed with nested braces
    start = text.rfind(r'\boxed{')
    if start == -1:
        return ''
    depth, i = 0, start + len(r'\boxed{')
    content_start = i
    while i < len(text):
        if text[i] == '{': depth += 1
        elif text[i] == '}':
            if depth == 0:
                return text[content_start:i].strip()
            depth -= 1
        i += 1
    return ''


def normalize_answer(ans: str) -> str:
    """Normalize answer string for comparison."""
    ans = ans.strip().lower()
    # Remove spaces and common formatting
    ans = ans.replace(' ', '').replace(',', '')
    # Normalize common fraction formats
    ans = ans.replace('\\frac{4}{3}', '4/3').replace('\\frac', '')
    # Try numeric comparison
    try:
        return str(round(float(ans), 4))
    except:
        return ans


def math_reward_fn(completions, answer, **kwargs):
    """
    Binary reward function — exactly as described in paper Section 2.
    Returns 1.0 if model answer matches ground truth, else 0.0

    Args:
        completions: list of model-generated strings
        answer     : list of ground truth answer strings
    Returns:
        list of float rewards (0.0 or 1.0)
    """
    rewards = []
    for completion, gt in zip(completions, answer):
        predicted = extract_boxed_answer(completion)
        # Try exact match first
        if normalize_answer(predicted) == normalize_answer(gt):
            rewards.append(1.0)
            continue
        # Try numeric tolerance match (for floating point answers)
        try:
            pred_val = float(predicted.replace(',', ''))
            gt_val   = float(gt.replace(',', ''))
            rewards.append(1.0 if abs(pred_val - gt_val) < 0.01 else 0.0)
        except:
            rewards.append(0.0)
    return rewards


# Quick test
test_completions = [
    'The answer is \\boxed{12.8} miles per hour.',
    'So the velocity is \\boxed{12.7}.',
    'I get \\boxed{10}.',
]
test_answers = ['12.8', '12.8', '12.8']
test_rewards = math_reward_fn(test_completions, test_answers)

print('✅ Reward function test:')
for comp, rew in zip(test_completions, test_rewards):
    extracted = extract_boxed_answer(comp)
    print(f'   Extracted: "{extracted}" → reward = {rew}')

✅ Reward function test:
   Extracted: "12.8" → reward = 1.0
   Extracted: "12.7" → reward = 0.0
   Extracted: "10" → reward = 0.0


## Cell 8: Load Model in 4-bit
This is the **key difference** from the paper — 4-bit QLoRA instead of full fp16.

In [ ]:
# ── Load model in 4-bit quantization ──────────────────────────────────────
# NOVEL: Paper used full fp16. We use 4-bit to fit T4.

def get_gpu_memory_gb():
    return torch.cuda.memory_allocated() / 1e9

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_name'],
    trust_remote_code=True,
    padding_side='left',   # important for batch generation
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = CFG['load_in_4bit'],
    bnb_4bit_quant_type       = CFG['quant_type'],       # nf4
    bnb_4bit_compute_dtype    = CFG['compute_dtype'],    # fp16
    bnb_4bit_use_double_quant = CFG['double_quant'],     
)

print('Loading model in 4-bit (this may take 2-3 minutes)...')
mem_before = get_gpu_memory_gb()

model = AutoModelForCausalLM.from_pretrained(
    CFG['model_name'],
    quantization_config = bnb_config,
    device_map          = 'auto',         
    trust_remote_code   = True,
    torch_dtype         = torch.float16,
)

mem_after = get_gpu_memory_gb()
print(f'✅ Model loaded')
print(f'   GPU memory used by model : {mem_after - mem_before:.2f} GB')
print(f'   Total GPU memory used    : {mem_after:.2f} GB')
print(f'   GPU memory free          : {(torch.cuda.get_device_properties(0).total_memory / 1e9) - mem_after:.2f} GB')

Loading tokenizer...
Loading model in 4-bit (this may take 2-3 minutes)...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

✅ Model loaded
   GPU memory used by model : 0.71 GB
   Total GPU memory used    : 0.71 GB
   GPU memory free          : 14.93 GB


## Cell 9: Baseline Evaluation on MATH500

We evaluate the base model on the **full MATH500 dataset** — exactly as the paper does.

MATH500 is a 500-problem curated subset of the MATH benchmark test set, covering 7 subject categories.

> **Time warning**: 500 problems × ~5s each on T4 ≈ 40-45 minutes.  
> For a quick sanity check first, change `dataset=math500` to `dataset=math500.select(range(50))`

Paper reports **36.0%** for Qwen2.5-Math-1.5B baseline on MATH500.


In [ ]:
# ── MATH500 Evaluation — matches paper exactly ────────────────────────────
# Paper uses the official Qwen2.5-Math evaluation pipeline on MATH500
# Dataset: HuggingFaceH4/MATH-500 (500 problems from MATH test set)
# Same 7 subject categories as paper Table 3

from datasets import load_dataset
import re, torch
from tqdm import tqdm

# ── Load MATH500 ───────────────────────────────────────────────────────────
print("Loading MATH500 dataset...")
math500 = load_dataset("HuggingFaceH4/MATH-500", split="test")
print(f"Loaded {len(math500)} problems")
print(f"Subjects: {sorted(set(math500['subject']))}")

# ── Prompt template — same as Qwen2.5-Math paper ──────────────────────────
QWEN_MATH_TEMPLATE = (
    "Please reason step by step, and put your final answer within \\boxed{{}}.\n\n"
    "Problem: {problem}"
)

# ── Answer normalisation (handles LaTeX fractions, decimals, etc.) ─────────
def normalise_math_answer(ans: str) -> str:
    """
    Normalise a LaTeX/text math answer for comparison.
    Handles common forms the model and dataset both produce.
    """
    if not ans:
        return ""
    ans = ans.strip()

    # Remove surrounding $...$
    ans = re.sub(r'^\$+|\$+$', '', ans).strip()

    # Normalise LaTeX fractions: \frac{a}{b} → a/b
    ans = re.sub(r'\\frac\{([^}]+)\}\{([^}]+)\}', r'\1/\2', ans)

    # Remove LaTeX formatting
    for cmd in [r'\\left', r'\\right', r'\\,', r'\\!', r'\\ ', r'\\text\{[^}]*\}']:
        ans = re.sub(cmd, '', ans)
    ans = re.sub(r'\\(sqrt|cdot|times|pm|approx)', r'\\\1', ans)

    # Normalise spaces and case
    ans = ans.replace(' ', '').lower()

    # Try numeric normalisation
    try:
        val = float(ans.replace(',', ''))
        # Round to 4 dp to handle floating point drift
        return str(round(val, 4))
    except ValueError:
        return ans


def extract_boxed_answer(text: str) -> str:
    """Extract the last \\boxed{...} content, handling nested braces."""
    # Simple non-nested first
    matches = re.findall(r'\\boxed\{([^{}]*)\}', text)
    if matches:
        return matches[-1].strip()
    # Nested brace fallback
    start = text.rfind(r'\boxed{')
    if start == -1:
        return ''
    depth, i = 0, start + len(r'\boxed{')
    content_start = i
    while i < len(text):
        if   text[i] == '{': depth += 1
        elif text[i] == '}':
            if depth == 0:
                return text[content_start:i].strip()
            depth -= 1
        i += 1
    return ''


def answers_match(predicted: str, ground_truth: str) -> bool:
    """Check if two answers are equivalent."""
    pred_norm = normalise_math_answer(predicted)
    gt_norm   = normalise_math_answer(ground_truth)

    if pred_norm == gt_norm:
        return True

    # Numeric tolerance (handles 12.7 vs 12.8 case from paper footnote 4)
    try:
        return abs(float(pred_norm) - float(gt_norm)) < 0.15
    except ValueError:
        return False


# ── Main evaluation function ───────────────────────────────────────────────
def evaluate_math500(model, tokenizer, dataset=math500,
                     max_new_tokens=256, desc="MATH500 Evaluation",
                     batch_size=1):
    """
    Evaluate model on full MATH500 — matches paper setup.

    Returns:
        overall_acc  : float  (0-100)
        subject_accs : dict   subject → accuracy
        results      : list   per-problem dicts
    """
    model.eval()
    results = []

    subject_correct = {}
    subject_total   = {}

    print(f"\n{'='*60}")
    print(f"{desc}")
    print(f"Evaluating {len(dataset)} problems | max_new_tokens={max_new_tokens}")
    print(f"{'='*60}\n")

    for i, item in enumerate(tqdm(dataset, desc=desc)):
        problem  = item['problem']
        gt       = item['answer']       
        subject  = item['subject']
        level    = item.get('level', '?')

        prompt = QWEN_MATH_TEMPLATE.format(problem=problem)

        inputs = tokenizer(
            prompt,
            return_tensors  = 'pt',
            truncation      = True,
            max_length      = 512,      
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = max_new_tokens,
                do_sample      = False,          
                pad_token_id   = tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        response   = tokenizer.decode(new_tokens, skip_special_tokens=True)
        predicted  = extract_boxed_answer(response)
        correct    = answers_match(predicted, gt)

        # Track per-subject
        subject_correct[subject] = subject_correct.get(subject, 0) + int(correct)
        subject_total[subject]   = subject_total.get(subject, 0) + 1

        results.append({
            'problem'  : problem,
            'gt'       : gt,
            'predicted': predicted,
            'correct'  : correct,
            'subject'  : subject,
            'level'    : level,
        })

    # ── Summary ────────────────────────────────────────────────────────────
    total_correct = sum(r['correct'] for r in results)
    overall_acc   = total_correct / len(results) * 100

    subject_accs  = {
        subj: subject_correct[subj] / subject_total[subj] * 100
        for subj in sorted(subject_total)
    }

    print(f"\n{'='*60}")
    print(f"RESULTS: {desc}")
    print(f"{'='*60}")
    print(f"  Overall accuracy : {total_correct}/{len(results)} = {overall_acc:.1f}%")
    print(f"  (Paper baseline  : 36.0%  |  Paper post-1shot: 73.6%)")
    print()

    # Paper Table 3 subject abbreviations
    abbrev = {
        'Algebra'               : 'Alg.',
        'Counting & Probability': 'C.P.',
        'Geometry'              : 'Geo.',
        'Intermediate Algebra'  : 'I.Alg.',
        'Number Theory'         : 'N.T.',
        'Prealgebra'            : 'Prealg.',
        'Precalculus'           : 'Precal.',
    }
    print(f"  {'Subject':<28} {'Ours':>8}   {'Paper base':>10}")
    print(f"  {'-'*50}")
    paper_base = {           
        'Algebra'               : 37.1,
        'Counting & Probability': 31.6,
        'Geometry'              : 39.0,
        'Intermediate Algebra'  : 43.3,
        'Number Theory'         : 24.2,
        'Prealgebra'            : 36.6,
        'Precalculus'           : 33.9,
    }
    for subj, acc in subject_accs.items():
        ab  = abbrev.get(subj, subj[:7])
        ref = paper_base.get(subj, 0.0)
        print(f"  {ab:<28} {acc:>7.1f}%   {ref:>9.1f}%")

    print(f"{'='*60}\n")
    return overall_acc, subject_accs, results


# ── Run baseline evaluation ────────────────────────────────────────────────

print("Starting MATH500 baseline evaluation...")
print("Tip: use math500.select(range(50)) for a quick 50-problem sanity check first\n")

baseline_acc, baseline_subject_accs, baseline_results = evaluate_math500(
    model, tokenizer,
    # dataset       = math500,
    dataset = math500.select(range(200)),
    max_new_tokens= 256,
    desc          = "BASELINE — Before 1-Shot RLVR",
)

baseline_accuracy = baseline_acc   # keep for later cells
print(f"📊 Baseline MATH500 accuracy: {baseline_accuracy:.1f}%")
print(f"   Paper reports: 36.0% for Qwen2.5-Math-1.5B")


Loading MATH500 dataset...
Loaded 500 problems
Subjects: ['Algebra', 'Counting & Probability', 'Geometry', 'Intermediate Algebra', 'Number Theory', 'Prealgebra', 'Precalculus']
Starting MATH500 baseline evaluation...
Tip: use math500.select(range(50)) for a quick 50-problem sanity check first


BASELINE — Before 1-Shot RLVR
Evaluating 200 problems | max_new_tokens=256




BASELINE — Before 1-Shot RLVR: 100%|██████████| 200/200 [49:05<00:00, 14.73s/it]


RESULTS: BASELINE — Before 1-Shot RLVR
  Overall accuracy : 21/200 = 10.5%
  (Paper baseline  : 36.0%  |  Paper post-1shot: 73.6%)

  Subject                          Ours   Paper base
  --------------------------------------------------
  Alg.                            13.5%        37.1%
  C.P.                             6.7%        31.6%
  Geo.                            12.5%        39.0%
  I.Alg.                           2.4%        43.3%
  N.T.                             4.5%        24.2%
  Prealg.                         28.1%        36.6%
  Precal.                          0.0%        33.9%

📊 Baseline MATH500 accuracy: 10.5%
   Paper reports: 36.0% for Qwen2.5-Math-1.5B
